# SAPS-II / Severity & Mortality Analysis

Notebook für die Folie **„Value of SAPS II“** und Korrekturfaktoren (Schweregrad, Beatmung).

## Scores & Time Windows

| Score | Time Window | Source |
|-------|-------------|--------|
| **SAPS II** | First 24h of ICU stay | `mimiciii_derived.sapsii` (first-day definition) |
| **SOFA total** | First 24h of ICU stay | `mimiciii_derived.sofa` (first-day definition) |

All severity scores in this notebook refer to the **first 24 hours** of the ICU stay. When comparing across intervention timing groups, keep in mind that "first 24h" may already include time after the intervention started.

For alternative time windows, use `compute_sofa_from_raw()` or `get_labs_for_window()` from `src/utils.py`.

In [ ]:
# --- Setup: Pfad und Imports ---
import sys
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Projektwurzel = report_abgabe (ein Ordner über notebooks/)
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
from src.db_connect import get_engine

engine = get_engine()

In [3]:
# --- Optional: Tabellen im Schema prüfen ---
# pd.read_sql("SELECT table_schema, table_name FROM information_schema.tables WHERE table_schema IN ('mimiciii_derived','public') ORDER BY 1,2", engine)

## Daten laden

SAPS II aus `mimiciii_derived.sapsii`, Kohorte aus `cohort_respiratory` (respiratorische Diagnose + Beatmung). Schema: **mimiciii_derived** (nicht mimic_derived).

In [4]:
# --- Daten laden: SAPS II + Kohorte (cohort_respiratory) ---
# Voraussetzung: build_7_views.sql und sapsii.sql ausgeführt, ggf. t_create_cohort_respiratory.sql
query = """
SELECT c.hadm_id, c.outcome_death, c.intervention_ventilation, s.sapsii
FROM cohort_respiratory c
LEFT JOIN icustays i ON i.hadm_id = c.hadm_id
LEFT JOIN mimiciii_derived.sapsii s ON s.icustay_id = i.icustay_id
WHERE s.sapsii IS NOT NULL
"""
# Falls mehrere ICU-Stays pro hadm_id: ersten Stay pro Aufnahme behalten
df_severity = pd.read_sql(query, engine).drop_duplicates(subset=["hadm_id"], keep="first")
df_severity["outcome_death"] = df_severity["outcome_death"].astype(int)
print(f"Daten geladen. Anzahl Aufnahmen: {len(df_severity)}")
display(df_severity.head())

            table_name
0              d_items
1           admissions
2              callout
3           caregivers
4        chartevents_1
5        chartevents_2
6        chartevents_3
7        chartevents_4
8        chartevents_5
9        chartevents_6
10       chartevents_7
11       chartevents_8
12       chartevents_9
13      chartevents_10
14      chartevents_11
15      chartevents_12
16      chartevents_13
17      chartevents_14
18      chartevents_15
19      chartevents_16
20         chartevents
21      chartevents_17
22           cptevents
23      datetimeevents
24       diagnoses_icd
25            drgcodes
26               d_cpt
27     d_icd_diagnoses
28    d_icd_procedures
29          d_labitems
30            icustays
31      inputevents_cv
32      inputevents_mv
33           labevents
34  microbiologyevents
35          noteevents
36        outputevents
37            patients
38       prescriptions
39  procedureevents_mv
40      procedures_icd
41            services
42         

## Auswertung: Mortalität nach SAPS-II-Gruppe und Beatmung

In [ ]:
# Nutze die load_sql Funktion (wie Simon), um die neue SQL-Datei auszuführen
df_severity = load_sql('../sql/get_severity_stats.sql')

print(f"Daten geladen. Anzahl Patienten: {len(df_severity)}")
display(df_severity.head())

ProgrammingError: (psycopg2.errors.UndefinedTable) FEHLER:  Relation »mimic_derived.sapsii« existiert nicht
LINE 10: LEFT JOIN mimic_derived.sapsii s ON c.hadm_id = s.hadm_id;
                   ^

[SQL: SELECT 
    c.subject_id,
    c.hadm_id,
    c.gender,
    c.age,
    c.intervention_ventilation, 
    c.outcome_death,
    s.sapsii
FROM cohort_respiratory c
LEFT JOIN mimic_derived.sapsii s ON c.hadm_id = s.hadm_id;]
(Background on this error at: https://sqlalche.me/e/20/f405)

In [ ]:
# 1. Gruppen bilden (SAPS II in Kategorien einteilen)
df_severity['saps_group'] = pd.cut(df_severity['sapsii'], 
                                   bins=[0, 20, 40, 60, 100], 
                                   labels=['0-20 (Leicht)', '21-40 (Mittel)', '41-60 (Schwer)', '>60 (Sehr Schwer)'])

# 2. Grafik erstellen
plt.figure(figsize=(12, 6))
sns.barplot(data=df_severity, x='saps_group', y='outcome_death', hue='intervention_ventilation', palette='magma')

plt.title('Sterberate nach Krankheitsschwere (SAPS II, first 24h) und Beatmung')
plt.ylabel('Mortalitätsrate (0.0 bis 1.0)')
plt.xlabel('SAPS II Gruppe')
plt.legend(title='Beatmung', labels=['Nein', 'Ja'])
plt.show()

NameError: name 'df_severity' is not defined